In [ ]:
import os
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader

from cbir.dataset import CbirDataset
from cbir.models import DatabaseFeatureExtractor

In [ ]:
BATCH_SIZE = 60
OXFORD_ROOT = "/home/ubuntu/data/datasets/roxford5k/jpg"
PARIS_ROOT = "/home/ubuntu/data/datasets/rparis6k/jpg"
CACHE_DIR = "/home/ubuntu/data/feature_cache"

In [ ]:
oxford_dataset = CbirDataset(OXFORD_ROOT)
oxford_dataloader = DataLoader(oxford_dataset, batch_size=BATCH_SIZE)
print(f"Number of batches in roxford5k: {len(oxford_dataloader)}")

paris_dataset = CbirDataset(PARIS_ROOT)
paris_dataloader = DataLoader(paris_dataset, batch_size=BATCH_SIZE)
print(f"Number of batches in rparis6k: {len(paris_dataloader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
db_extractor = DatabaseFeatureExtractor(num_features=500, num_clusters=10).to(device)
db_extractor.eval()

In [ ]:
with torch.no_grad():
    print(f"Processing roxford5k....")

    oxford_feature_cache = {}

    for batch_images, batch_ids in tqdm(oxford_dataloader):
        batch_images = batch_images.to(device)
        batched_features = db_extractor(batch_images)
        batched_features = batched_features.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]
            single_image_features = batched_features[i]            
            oxford_feature_cache[image_id] = single_image_features

    torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "roxford5k_features.pkl"))
    print(f"Features extracted successfully and saved to {CACHE_DIR}/roxford5k_features.pkl")

    print(f"Processing rparis6k....")

    paris_feature_cache = {}

    for batch_images, batch_ids in tqdm(paris_dataloader):
        batch_images = batch_images.to(device)
        batched_features = db_extractor(batch_images)
        batched_features = batched_features.cpu()

        for i in range(len(batch_ids)):
            image_id = batch_ids[i]
            single_image_features = batched_features[i]            
            paris_feature_cache[image_id] = single_image_features

    torch.save(oxford_feature_cache, os.path.join(CACHE_DIR, "rparis6k_features.pkl"))
    print(f"Features extracted successfully and saved to {CACHE_DIR}/rparis6k_features.pkl")